In [3]:
import pandas as pd

#EDA

In [ ]:
df=pd.read_csv('Data/train-test.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB


In [6]:
print("Shape:", df.shape)

Shape: (48000, 14)


In [7]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [8]:
df.isnull().sum()

,0
load_id,0
pickup,0
delivery,0
pickup_lat,0
pickup_lon,0
delivery_lat,0
delivery_lon,0
distance,0
equipment,0
weight,300


In [9]:
df['equipment'].value_counts()

,count
equipment,
Dry Van,27202
Reefer,12045
Flatbed,8753


In [10]:
(df['weight'] < 0).sum()

np.int64(292)

In [11]:
df[df["weight"] < 0]["weight"].value_counts()

,count
weight,
-47500.0,13
-19389.0,2
-33033.0,2
-25982.0,2
-29838.0,2
...,...
-27099.0,1
-16229.0,1
-35664.0,1


In [12]:
df[df["weight"] <= 0]["weight"].describe()

,weight
count,292.000000
mean,-31724.195205
std,8262.536326
min,-47500.000000
25%,-37284.250000
50%,-31821.500000
75%,-25928.500000
max,-5000.000000


In [13]:
negative = df[df["weight"] < 0]

negative[[
    "pickup",
    "delivery",
    "equipment",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "posted_rate"
]].describe(include="all")

,pickup,delivery,equipment,distance,weight,market_index,quote_signal,posted_rate
count,292,292,292,292.000000,292.000000,289.000000,292.000000,292.000000
unique,64,62,3,NaN,NaN,NaN,NaN,NaN
top,Cincinnati,Baton Rouge,Dry Van,NaN,NaN,NaN,NaN,NaN
freq,11,10,155,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,1120.920205,-31724.195205,1.096322,2.041966,2389.786233
std,NaN,NaN,NaN,733.132047,8262.536326,0.171562,0.298395,1641.761285
min,NaN,NaN,NaN,75.100000,-47500.000000,0.789880,1.157730,247.370000
25%,NaN,NaN,NaN,531.975000,-37284.250000,0.961550,1.868555,1257.410000
50%,NaN,NaN,NaN,930.450000,-31821.500000,1.065710,2.033965,1994.960000
75%,NaN,NaN,NaN,1537.125000,-25928.500000,1.241320,2.194368,3185.257500


In [14]:
df.groupby(df["weight"] < 0)[
    ["distance", "market_index", "quote_signal", "posted_rate"]
].median()

,distance,market_index,quote_signal,posted_rate
weight,,,,
False,953.35,1.05577,2.055905,2031.16
True,930.45,1.06571,2.033965,1994.96


In [15]:
df.groupby("equipment")["weight"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
equipment,,,,,
Dry Van,27027,31027.293447,31373.0,-47500.0,47500.0
Flatbed,8698,30963.625776,31462.5,-47500.0,47500.0
Reefer,11975,31079.714572,31524.0,-47500.0,47500.0


# Note:
#### Negative weight values appear to follow a reversed encoding pattern.
#### For negative values, the cleaned weight is calculated as 47,500 + weight,
#### while positive values remain unchanged.

In [16]:
df["weight_clean"] = df["weight"]

negative_mask = df["weight"] < 0

df.loc[negative_mask, "weight_clean"] = (
    47500 + df.loc[negative_mask, "weight"]
)

In [17]:
df['weight_clean'].describe()

,weight_clean
count,47700.000000
mean,31319.619686
std,8088.996137
min,0.000000
25%,25813.000000
50%,31439.000000
75%,37018.250000
max,47500.000000


In [18]:
# Fill missing weight values with the median
median_weight = df["weight_clean"].median()

df["weight_clean"] = df["weight_clean"].fillna(median_weight)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
 14  weight_clean  48000 non-null  float64
dtypes: float64(10), object(5)
memory usage: 5.5+ MB


In [20]:
df[df["market_index"].isna()].describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate,weight_clean
count,374.000000,374.000000,374.000000,374.000000,374.000000,373.000000,0.0,374.000000,374.000000,374.000000
mean,35.742501,-90.229000,35.836751,-91.496143,1164.747861,30001.423592,NaN,2.072137,2392.115107,30386.283422
std,4.314111,13.364178,4.165810,14.223787,751.243031,9363.657800,NaN,0.298629,1434.640755,7989.424750
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-25804.000000,NaN,0.985800,207.710000,7483.000000
25%,32.546280,-94.785560,33.079210,-98.580970,553.525000,25304.000000,NaN,1.897158,1254.287500,25310.500000
50%,35.297920,-87.528710,35.373760,-88.089150,991.450000,30608.000000,NaN,2.057835,2081.100000,30642.000000
75%,39.548380,-82.299940,39.548380,-82.783790,1626.725000,35951.000000,NaN,2.228860,3294.790000,35950.500000
max,44.302960,-69.500000,44.302960,-69.500000,3163.400000,47500.000000,NaN,3.153980,9454.940000,47500.000000


In [21]:
df[df["market_index"].isna()]["date"].describe()

,date
count,374
unique,209
top,2025-06-15
freq,5


In [22]:
df[df["market_index"].isna()]["equipment"].value_counts()

,count
equipment,
Dry Van,202
Reefer,99
Flatbed,73


In [23]:
df[df["market_index"].isna()]["date"].value_counts()

,count
date,
2025-06-15,5
2025-07-27,5
2025-04-15,4
2025-03-28,4
2025-10-02,4
...,...
2025-10-10,1
2025-10-25,1
2025-10-23,1


In [24]:
# Calculate the daily average market index
daily_market_mean = df.groupby("date")["market_index"].transform("mean")

# Fill missing values with the average of the same day
df["market_index"] = df["market_index"].fillna(daily_market_mean)

In [25]:
df["market_index"].isna().sum()

np.int64(0)

In [26]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate,weight_clean
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,48000.000000,48000.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083365,2.062468,2373.980682,31320.365813
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168029,0.291391,1486.493245,8063.683364
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000,0.000000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000,25856.000000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000,31439.000000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219372,2.221685,3330.750000,36985.000000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000,47500.000000


#Feature Engineering

In [27]:
df.columns.tolist()

['load_id',
 'pickup',
 'delivery',
 'pickup_lat',
 'pickup_lon',
 'delivery_lat',
 'delivery_lon',
 'distance',
 'equipment',
 'weight',
 'date',
 'market_index',
 'quote_signal',
 'posted_rate',
 'weight_clean']

In [28]:
df["date"] = pd.to_datetime(df["date"])
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['week'] = df['date'].dt.isocalendar().week.astype(int)


In [29]:
df = df.drop(columns=["load_id",
#  'pickup_lat',
#  'pickup_lon',
#  'delivery_lat',
#  'delivery_lon',
                    'weight'])

In [30]:
numerical_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight_clean",
    "market_index",
    "quote_signal",
    "month",
    "day"
]

categorical_features = [
    "pickup",
    "delivery",
    "equipment"
]

#Visualization

In [33]:
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

In [34]:
df = df.copy()

TARGET = "posted_rate"

In [36]:
fig = px.histogram(
    df,
    x=TARGET,
    nbins=80,
    marginal="box",
    title="Distribution of Posted Freight Rate",
    labels={TARGET: "Posted Rate"},
    hover_data=[TARGET]
)

fig.update_layout(
    template="plotly_white",
    height=550
)

fig.show()

In [37]:
fig = px.scatter(
    df,
    x="distance",
    y=TARGET,
    opacity=0.45,
    trendline="lowess",
    title="Distance vs Posted Freight Rate",
    labels={
        "distance": "Distance",
        TARGET: "Posted Rate"
    },
    hover_data=["load_id"] if "load_id" in df.columns else None
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

In [39]:
fig = px.scatter(
    df,
    x="weight_clean",
    y=TARGET,
    opacity=0.45,
    trendline="lowess",
    title="Weight vs Posted Freight Rate",
    labels={
        "weight_clean": "Weight",
        TARGET: "Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

In [41]:
corr_cols = [
    col for col in numerical_features + [TARGET]
    if col in df.columns
]

corr_matrix = df[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Matrix — Numerical Features",
    labels=dict(color="Correlation")
)

fig.update_layout(
    template="plotly_white",
    height=700
)

fig.show()

In [43]:
fig = px.violin(
    df,
    x="equipment",
    y=TARGET,
    color="equipment",
    box=True,
    points=False,
    title="Posted Rate Distribution by Equipment Type",
    labels={
        "equipment": "Equipment Type",
        TARGET: "Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=600,
    showlegend=False
)

fig.show()

In [45]:
top_destinations = (
    df["delivery"]
    .value_counts()
    .head(20)
    .index
)

df_destination = df[
    df["delivery"].isin(top_destinations)
]

fig = px.box(
    df_destination,
    x="delivery",
    y=TARGET,
    points=False,
    title="Posted Rate by Top 20 Destination Locations",
    labels={
        "delivery": "Destination",
        TARGET: "Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=650,
    xaxis_tickangle=-45
)

fig.show()

In [46]:
daily_rate = (
    df.groupby("date")[TARGET]
    .agg(["mean", "median", "count"])
    .reset_index()
)

fig = px.line(
    daily_rate,
    x="date",
    y="mean",
    title="Average Daily Posted Freight Rate",
    labels={
        "date": "Date",
        "mean": "Average Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=550
)

fig.show()

In [48]:
monthly_rate = (
    df.groupby("month")[TARGET]
    .agg(["mean", "median", "count"])
    .reset_index()
)

monthly_rate["month_name"] = pd.to_datetime(
    monthly_rate["month"],
    format="%m"
).dt.strftime("%b")

fig = px.bar(
    monthly_rate,
    x="month_name",
    y="mean",
    title="Average Posted Rate by Month",
    labels={
        "month_name": "Month",
        "mean": "Average Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=550
)

fig.show()

In [51]:
fig = px.box(
    df,
    x="month",
    y=TARGET,
    category_orders={
        "month": [
            "Jan", "Feb", "Mar", "Apr",
            "May", "Jun", "Jul", "Aug",
            "Sep", "Oct", "Nov", "Dec"
        ]
    },
    points=False,
    title="Posted Rate Distribution by Month",
    labels={
        "month_name": "Month",
        TARGET: "Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

In [53]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

fig = px.box(
    df,
    x="dayofweek",
    y=TARGET,
    category_orders={"day_of_week": day_order},
    points=False,
    title="Posted Rate Distribution by Day of Week",
    labels={
        "day_of_week": "Day of Week",
        TARGET: "Posted Rate"
    }
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

In [54]:
available_features = [
    col for col in [
        "distance",
        "weight",
        "market_index"
    ]
    if col in df.columns
]

fig = go.Figure()

for feature in available_features:

    fig.add_trace(
        go.Scatter(
            x=df[feature],
            y=df[TARGET],
            mode="markers",
            name=feature,
            visible=(feature == available_features[0]),
            opacity=0.4
        )
    )

buttons = []

for i, feature in enumerate(available_features):

    visibility = [False] * len(available_features)
    visibility[i] = True

    buttons.append(
        dict(
            label=feature,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": f"{feature} vs Posted Rate"
                }
            ]
        )
    )

fig.update_layout(
    title=f"{available_features[0]} vs Posted Rate",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.02,
            y=1
        )
    ],
    template="plotly_white",
    height=650
)

fig.show()